# 02 - Agrupamento (K-Means e K-Medoids)
**Objetivo:** descobrir perfis de gravidade/porte dos acidentes a partir de variáveis numéricas.
Pré-requisito: rodar o notebook 01 (gera `data/base_acidentes.pkl`).

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = RAIZ / "data"; FIG = RAIZ / "figuras"; FIG.mkdir(exist_ok=True)
ANO = 2023
SEED = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
try:
    from sklearn_extra.cluster import KMedoids
    print("KMedoids: scikit-learn-extra")
except Exception as e:
    print("scikit-learn-extra indisponível, usando pacote 'kmedoids':", e)
    import kmedoids
    class KMedoids:
        def __init__(self, n_clusters, random_state=42, **kw): self.k, self.rs = n_clusters, random_state
        def fit(self, X):
            from sklearn.metrics import pairwise_distances
            r = kmedoids.fasterpam(pairwise_distances(X), self.k, random_state=self.rs)
            self.labels_ = np.array(r.labels); return self
base = pd.read_pickle(DATA / "base_acidentes.pkl")
NUM = ["pessoas", "veiculos", "mortos", "feridos_leves", "feridos_graves"]   # NÃO CONFIRMADO
print(base.shape)

## Pré-processamento
- **log1p**: contagens têm cauda longa (muitos acidentes com 0 mortos, poucos com vários); o log evita que poucos casos extremos dominem a distância.
- **StandardScaler**: K-Means e K-Medoids usam distância euclidiana, então as variáveis precisam estar na mesma escala.

In [ ]:
print("Assimetria (antes):"); print(base[NUM].skew().round(2))
X = np.log1p(base[NUM])
print("Assimetria (depois do log1p):"); print(X.skew().round(2))
Xs = StandardScaler().fit_transform(X)

## Escolha do algoritmo e do k (2 a 10)
K-Medoids exige matriz de distâncias (n²), inviável com dezenas de milhares de linhas. Por isso a **comparação** usa uma **amostra aleatória de 10.000 acidentes** (`random_state=42`). A silhueta também é calculada na amostra.

In [ ]:
N_AMOSTRA = min(10000, len(Xs))
idx = np.random.RandomState(SEED).choice(len(Xs), N_AMOSTRA, replace=False)
Xa = Xs[idx]
res = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xa)
    md = KMedoids(n_clusters=k, random_state=SEED).fit(Xa)
    res.append({"k": k, "inercia_kmeans": km.inertia_,
                "sil_kmeans": silhouette_score(Xa, km.labels_),
                "sil_kmedoids": silhouette_score(Xa, md.labels_)})
    print(res[-1])
res = pd.DataFrame(res).set_index("k"); display(res.round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
res[["sil_kmeans", "sil_kmedoids"]].plot(marker="o", ax=ax[0]); ax[0].set_title("Silhueta por k"); ax[0].set_ylabel("Silhueta média")
res["inercia_kmeans"].plot(marker="o", ax=ax[1], color="gray"); ax[1].set_title("Método do cotovelo (K-Means)"); ax[1].set_ylabel("Inércia")
plt.tight_layout(); plt.savefig(FIG / "cluster_silhueta_cotovelo.png", dpi=150); plt.show()

## Decisão (preencher olhando os gráficos)
Escreva aqui a justificativa: qual algoritmo teve melhor silhueta, onde o cotovelo dobra e se os grupos fazem sentido para interpretar. Evite escolher k=2 só pela silhueta se ele não separa nada útil.

In [ ]:
ALGORITMO = "kmeans"   # "kmeans" ou "kmedoids"  <- AJUSTE conforme a tabela acima
K = 4                  # <- AJUSTE conforme silhueta + cotovelo + interpretação
if ALGORITMO == "kmeans":
    modelo = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(Xs)
    base["grupo"] = modelo.labels_
else:
    # K-Medoids na amostra; o restante é atribuído ao medoide mais próximo
    md = KMedoids(n_clusters=K, random_state=SEED).fit(Xa)
    centros = np.vstack([Xa[md.labels_ == g].mean(axis=0) for g in range(K)])
    base["grupo"] = np.argmin(((Xs[:, None, :] - centros[None]) ** 2).sum(-1), axis=1)
print(base["grupo"].value_counts().sort_index())
print("Silhueta na amostra:", round(silhouette_score(Xs[idx], base["grupo"].values[idx]), 3))

In [ ]:
perfil = base.groupby("grupo")[NUM].mean().round(2)
perfil["n_acidentes"] = base["grupo"].value_counts().sort_index()
perfil["% do total"] = (perfil["n_acidentes"] / len(base) * 100).round(1)
perfil["% com morte"] = (base.assign(m=base["mortos"] > 0).groupby("grupo")["m"].mean() * 100).round(1)
display(perfil)

## Nomear os grupos
Depois de ver a tabela, dê nomes (ex.: 'Leves com 1 veículo', 'Graves com vários feridos', 'Fatais'...).

In [ ]:
NOMES = {0: "Leves, poucos envolvidos", 1: "Fatais", 2: "Feridos graves", 3: "Feridos leves, muitos envolvidos"}
base["grupo_nome"] = base["grupo"].map(NOMES).fillna(base["grupo"].astype(str))
perfil_z = pd.DataFrame(Xs, columns=NUM).groupby(base["grupo"].values).mean()
plt.figure(figsize=(8, 4)); sns.heatmap(perfil_z, annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Perfil dos grupos (média padronizada)"); plt.xlabel(""); plt.ylabel("Grupo")
plt.tight_layout(); plt.savefig(FIG / "cluster_perfil.png", dpi=150); plt.show()

In [ ]:
pca = PCA(n_components=2, random_state=SEED).fit(Xs)
P = pca.transform(Xs[idx])
plt.figure(figsize=(7, 5))
sns.scatterplot(x=P[:, 0], y=P[:, 1], hue=base["grupo"].values[idx], palette="tab10", s=10, alpha=.6)
plt.title(f"PCA 2D (variância explicada: {pca.explained_variance_ratio_.sum()*100:.1f}%)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.tight_layout(); plt.savefig(FIG / "cluster_pca.png", dpi=150); plt.show()
base.to_pickle(DATA / "base_acidentes.pkl")